In [ ]:
import json

import duckdb
import pandas as pd

from ledger.config import load_settings
from ledger.extract import raw_path
from ledger.frames import districts, read_raw

pd.set_option("display.max_columns", 50, "display.width", 200)
s = load_settings()
paths = {name: str(raw_path(s, src)) for name, src in s.sources.items()}
manifest = json.loads((s.raw_dir / "manifest.json").read_text())
{k: (v["version"], v["rows"]) for k, v in manifest.items()}

In [ ]:
con = duckdb.connect()
con.sql(f"SUMMARIZE SELECT * FROM read_csv('{paths['tcl_change']}')").df()

In [ ]:
chg = read_raw(s, "tcl_change")
summ = read_raw(s, "tcl_summary")
viirs = read_raw(s, "viirs_weekly")
names = districts(read_raw(s, "boundaries"))
chg.shape, summ.shape, viirs.shape, names.shape

In [ ]:
for col in ["threshold", "is_primary", "driver", "iucn_cat", "plantation_type"]:
    print(col, chg[col].value_counts(dropna=False).to_dict(), "\n")

In [ ]:
print(chg["adm1"].nunique(), "states with any loss")
print(chg[["adm1", "adm2"]].drop_duplicates().shape[0], "districts with any loss")
print(names["state"].nunique(), "states in GADM;", len(names), "districts in GADM")
names.groupby("state").size().sort_values(ascending=False).head(10)